# Gemma 4 on Amazon Bedrock Mantle — end to end

Google DeepMind's Gemma 4 family (Apache 2.0) on the `bedrock-mantle` endpoint:
simple inference → streaming → reasoning → stateful chat → tool use →
structured JSON → multimodal → production hardening.

**Three variants, interleaved throughout this notebook:**

| Model ID | Architecture | Params | Context |
|---|---|---|---|
| `google.gemma-4-31b` | Dense | 30.7B | 256K |
| `google.gemma-4-26b-a4b` | Mixture-of-Experts | 25.2B total / 3.8B active | 256K |
| `google.gemma-4-e2b` | Dense (Per-Layer Embeddings) | 5.1B total / 2.3B effective | 128K |

**Gemma 4 is `bedrock-mantle`-only.** There is no `bedrock-runtime` support —
`invoke_model` and `converse` return errors for these model IDs.

## Self-contained, but see also
This notebook stands alone. For deeper background:
- **Auth, the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention/ZDR, CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retries, service tiers, TTFT (time-to-first-token) benchmarking** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

You also need AWS credentials with `bedrock-mantle:CreateInference` and
`bedrock-mantle:CallWithBearerToken` (the managed policy
`AmazonBedrockMantleInferenceAccess` grants both).

In [1]:
import base64
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import (
    err,
    function_calls,
    parse_json_lenient,
    post,
    redact_ids,
    response_text,
    safe_print,
    stream_lines,
)

# Gemma 4 is available in ALL FOUR mantle Regions — the only family that is.
REGION = "us-east-1"

DENSE = "google.gemma-4-31b"
MOE = "google.gemma-4-26b-a4b"
COMPACT = "google.gemma-4-e2b"

# NOTE the "/openai" prefix. Gemma 4's model card calls this out explicitly:
# its paths differ from the bare /v1 used by most other mantle models.
PREFIX = "/openai/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)

base URL: https://bedrock-mantle.us-east-1.api.aws/openai/v1


## 1. Simple inference with the Responses API

The Responses API is AWS's recommended surface for new applications, and for
Gemma 4 specifically it is the **only** way to read reasoning output (see §3).

Auth here uses a short-term Bedrock API key minted from ambient IAM credentials.
(Full explanation, including the SigV4 (AWS Signature Version 4) alternative that
needs no key at all, is
in `../00-foundations/01`.)

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token. Tokens last <=12h and cannot be refreshed,
# so don't construct one at import time and reuse it for hours.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

response = client.responses.create(
    model=DENSE,
    input="Explain what a mixture-of-experts model is, in two sentences.",
    max_output_tokens=200,
)
print(response.output_text)

A Mixture-of-Experts (MoE) model is a neural network architecture that replaces a single large layer with multiple specialized "expert" subnetworks. A gating mechanism dynamically selects only the most relevant experts to process each specific input, allowing the model to increase its total parameter count without significantly increasing the computational cost per token.


## 2. Sampling parameters — the Gemma 4 trap

This is the single most surprising thing about Gemma 4 on the Responses API:

1. **`temperature` accepts only its default value, `1.0`.** Any other value —
   including `0.0` and `0.7` — is rejected with
   *"Unsupported parameter: 'temperature' is not supported with this model."*
2. **`top_p` is rejected outright**, even though AWS's own launch blog recommends
   `top_p=0.95`. That guidance applies to Chat Completions, not Responses.

So on Responses you effectively cannot tune sampling for this model. Sweep the
values and see for yourself:

In [3]:
print(f"{'temperature':>12} {'HTTP':>6}  detail")
print("-" * 74)
for value in (0.0, 0.2, 0.5, 0.7, 0.9, 1.0, 1.5):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "Reply with exactly: OK",
            "max_output_tokens": 16,
            "temperature": value,
        },
        region=REGION,
    )
    print(f"{value:>12} {code:>6}  {'' if code == 200 else err(data)[:52]}")

code, data = post(
    f"{PREFIX}/responses",
    {"model": DENSE, "input": "Reply OK", "max_output_tokens": 16, "top_p": 0.95},
    region=REGION,
)
print(f"\n  top_p=0.95        -> HTTP {code} {err(data)[:70]}")

 temperature   HTTP  detail
--------------------------------------------------------------------------


         0.0    400  Unsupported parameter: 'temperature' is not supporte


         0.2    400  Unsupported parameter: 'temperature' is not supporte


         0.5    400  Unsupported parameter: 'temperature' is not supporte


         0.7    400  Unsupported parameter: 'temperature' is not supporte


         0.9    400  Unsupported parameter: 'temperature' is not supporte


         1.0    200  


         1.5    400  Unsupported parameter: 'temperature' is not supporte



  top_p=0.95        -> HTTP 400 Unsupported parameter: 'top_p' is not supported with this model.


Only `1.0` passes — which is Gemma 4's documented default. The pattern across
mantle's Responses API is that a model accepts `temperature` **only at its own
default** (Grok's default is `0.7`, so Grok rejects `1.0` and accepts `0.7`).

`1.0` also happens to be the value AWS and Google recommend anyway
(greedy decoding at `temperature=0` sends Gemma 4 into repetition loops involving
reserved vocabulary tokens), so the constraint pushes you towards the right
setting — but it means **you cannot lower the temperature for extraction tasks**
on this API.

The simplest safe approach on Responses: **omit both parameters** and accept the
defaults.

In [4]:
# The parameter surface on Chat Completions is NOT the same as on Responses, and
# it is not stable over time: on 12 Aug 2026 Gemma 4 tightened it to match
# current OpenAI semantics. So probe it rather than trusting a table -- including
# the tables in this notebook.
probes = [
    ("max_tokens=16", {"max_tokens": 16}),
    ("max_completion_tokens=16", {"max_completion_tokens": 16}),
    ("+ temperature=0.2", {"max_completion_tokens": 16, "temperature": 0.2}),
    ("+ temperature=1.0", {"max_completion_tokens": 16, "temperature": 1.0}),
    ("+ top_p=0.95", {"max_completion_tokens": 16, "top_p": 0.95}),
]
print(f"{"parameters":<26} {"HTTP":<5} detail")
print("-" * 76)
for label, extra in probes:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": DENSE,
            "messages": [{"role": "user", "content": "Reply OK"}],
            **extra,
        },
        region=REGION,
    )
    print(f"{label:<26} {code:<5} {"" if code == 200 else err(data)[:42]}")

print()
print("=> Gemma 4 takes max_completion_tokens here, not max_tokens, and pins")
print("   temperature to its default. top_p is rejected outright.")
print("   The Responses API rejects the same sampling parameters (section 2),")
print("   so switching API does NOT buy you sampling control on this model.")


parameters                 HTTP  detail
----------------------------------------------------------------------------


max_tokens=16              400   Unsupported parameter: 'max_tokens' is not


max_completion_tokens=16   200   


+ temperature=0.2          400   Unsupported value: 'temperature' does not 


+ temperature=1.0          200   


+ top_p=0.95               400   Unsupported parameter: 'top_p' is not supp

=> Gemma 4 takes max_completion_tokens here, not max_tokens, and pins
   temperature to its default. top_p is rejected outright.
   The Responses API rejects the same sampling parameters (section 2),
   so switching API does NOT buy you sampling control on this model.


In [5]:
# Another undocumented constraint: max_output_tokens has a MINIMUM of 16.
for n in (8, 15, 16):
    code, data = post(
        f"{PREFIX}/responses",
        {"model": DENSE, "input": "Hi", "max_output_tokens": n},
        region=REGION,
    )
    detail = "" if code == 200 else err(data)[:70]
    print(f"  max_output_tokens={n:3} -> HTTP {code} {detail}")

  max_output_tokens=  8 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


  max_output_tokens= 15 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


  max_output_tokens= 16 -> HTTP 200 


## 3. Reasoning — and why the API choice matters

All three variants have built-in reasoning. The critical detail from the model
card: reasoning effort is honoured on **both** Responses and Chat Completions,
and the model does the extended thinking either way — but **only the Responses
API returns the reasoning content**. On Chat Completions you pay for those
tokens and never see them.

In [6]:
resp = client.responses.create(
    model=DENSE,
    input=(
        "A train leaves at 3pm travelling 60 km/h. Another leaves an hour later "
        "at 90 km/h from the same station on the same track. When does the second "
        "catch the first?"
    ),
    reasoning={"effort": "high"},
    max_output_tokens=1200,
)

reasoning_blocks = []
for item in resp.output:
    if item.type == "reasoning":
        for block in item.content:
            text = getattr(block, "text", "")
            if text:
                reasoning_blocks.append(text)

print("=== REASONING (visible only on the Responses API) ===")
print(("\n".join(reasoning_blocks))[:700] or "(none returned)")
print("\n=== FINAL ANSWER ===")
print(resp.output_text[:400])
print("\nreasoning tokens:", resp.usage.output_tokens_details.reasoning_tokens)

=== REASONING (visible only on the Responses API) ===
*   Train 1: Starts at 3 PM, speed = 60 km/h.
    *   Train 2: Starts at 4 PM (one hour later), speed = 90 km/h.
    *   Both start from the same station on the same track.
    *   Goal: Find the time when Train 2 catches Train 1.

    *   $v_1 = 60$ km/h
    *   $v_2 = 90$ km/h
    *   Time difference ($\Delta t$) = 1 hour.

    *   By 4 PM (when Train 2 starts), Train 1 has been travelling for 1 hour.
    *   Distance of Train 1 at 4 PM: $60 \text{ km/h} \times 1 \text{ hour} = 60 \text{ km}$.

    *   The relative speed is the difference between their speeds: $90 \text{ km/h} - 60 \text{ km/h} = 30 \text{ km/h}$.
    *   This means Train 2 closes the gap at a rate of 30 km every hour.

  

=== FINAL ANSWER ===
The second train will catch the first at **6:00 PM**.

Here is the step-by-step breakdown:

1.  **Find the head start:**
    The first train leaves at 3:00 PM and the second leaves at 4:00 PM. In that one hour, the first tr

In [7]:
# Which effort values are valid? Probe rather than assume.
for effort in ("none", "minimal", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "2+2?",
            "max_output_tokens": 32,
            "reasoning": {"effort": effort},
        },
        region=REGION,
    )
    print(f"  effort={effort:8} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  effort=none     -> HTTP 200 


  effort=minimal  -> HTTP 400 Unsupported value: 'minimal' is not supported with the 'goog


  effort=low      -> HTTP 200 


  effort=medium   -> HTTP 200 


  effort=high     -> HTTP 200 


`minimal` is rejected; the valid ladder is `none` / `low` / `medium` / `high`.

**Variant-specific advice:** for `gemma-4-e2b`, set `effort="high"`. The smallest
variant reasons extensively by default, and high effort keeps that thinking in
the dedicated reasoning channel instead of leaking into the final answer.

In [8]:
for model in (COMPACT, DENSE):
    r = client.responses.create(
        model=model,
        input="If 3 shirts dry in 4 hours, how long for 9 shirts on the same line?",
        reasoning={"effort": "high"},
        max_output_tokens=600,
    )
    reasoning_tokens = r.usage.output_tokens_details.reasoning_tokens
    print(
        f"{model:26} reasoning_tokens={reasoning_tokens:5}  "
        f"answer={r.output_text[:90]!r}"
    )

google.gemma-4-e2b         reasoning_tokens=    0  answer='This is a problem based on the idea that all shirts are drying simultaneously on the same '


google.gemma-4-31b         reasoning_tokens=    0  answer='It will still take **4 hours**. \n\nSince the shirts are all on the line at the same time, t'


## 4. Streaming

Reasoning and answer text arrive on **separate event types** — that's what lets
you render a "thinking…" panel distinct from the answer.

In [9]:
stream = client.responses.create(
    model=DENSE,
    input="List three properties of a good distributed queue.",
    reasoning={"effort": "low"},
    max_output_tokens=400,
    stream=True,
)

event_counts = {}
print("--- live stream ---")
try:
    for event in stream:
        event_counts[event.type] = event_counts.get(event.type, 0) + 1
        if event.type == "response.reasoning_text.delta":
            print("\033[2m" + event.delta + "\033[0m", end="", flush=True)
        elif event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after the deltas above: {type(exc).__name__}]")
print("\n\n--- event types seen ---")
for name, count in sorted(event_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")

--- live stream ---


A

 good

 distributed

 queue

...

 producers

 to

 be

 thrott



--- event types seen ---
   400  response.output_text.delta
     1  response.created
     1  response.in_progress
     1  response.output_item.added
     1  response.content_part.added
     1  response.output_text.done
     1  response.content_part.done
     1  response.output_item.done
     1  response.incomplete


## 5. Multi-turn: two approaches

### (a) Send the history yourself
`input` accepts the same role/content array that Chat Completions calls
`messages`. Fully stateless — nothing is retained server-side.

In [10]:
conversation = [
    {"role": "system", "content": "You are terse. Answer in one short sentence."},
    {"role": "user", "content": "What is a MoE model?"},
]
first = client.responses.create(model=MOE, input=conversation, max_output_tokens=120)
print("assistant:", first.output_text)

conversation += [
    {"role": "assistant", "content": first.output_text},
    {"role": "user", "content": "And why is it cheaper to run?"},
]
second = client.responses.create(model=MOE, input=conversation, max_output_tokens=120)
print("assistant:", second.output_text)

assistant: A Mixture of Experts (MoE) model is a neural network architecture that uses a routing mechanism to activate only a subset of its specialized sub-networks for each input.


assistant: It is cheaper because it only activates a small fraction of its total parameters for each calculation, reducing computational cost.


**Important for Gemma 4:** append only the *final answers* to history, never the
reasoning items. AWS warns that replaying prior reasoning back to the model
degrades later turns. Keep reasoning in your logs, strip it from `input`.

### (b) Server-side state with `previous_response_id`
Bedrock rebuilds the context for you. Cheaper on input tokens for long chats —
but it requires `store=True`, which retains input and output for **30 days**
in-Region (encrypted, project-scoped).

In [11]:
turn1 = client.responses.create(
    model=DENSE,
    input="My favourite database is DynamoDB. Reply with just: noted.",
    max_output_tokens=32,
    store=True,
)
print("turn 1 id:", redact_ids(turn1.id))

turn2 = client.responses.create(
    model=DENSE,
    input="What is my favourite database?",
    previous_response_id=turn1.id,
    max_output_tokens=48,
)
print("turn 2   :", turn2.output_text)

turn 1 id: resp_56ufg573...


turn 2   : Your favourite database is DynamoDB.


In [12]:
# The privacy/convenience trade-off, made concrete.
private = client.responses.create(
    model=DENSE, input="Secret: 42. Reply: ok.", max_output_tokens=16, store=False
)
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "What was the secret?",
        "previous_response_id": private.id,
        "max_output_tokens": 32,
    },
    region=REGION,
)
print(f"chaining from a store=False response -> HTTP {code}")
print("message:", err(data)[:100])
print("\n=> Choose: server-side state (store=True) OR zero retention, not both.")

chaining from a store=False response -> HTTP 404
message: Response not found.

=> Choose: server-side state (store=True) OR zero retention, not both.


## 6. Retrieve, and run in the background

Stored responses can be fetched later, and long jobs can run detached.

In [13]:
code, fetched = post(
    f"{PREFIX}/responses/{turn1.id}", None, region=REGION, method="GET"
)
print(f"GET  -> {code} status={fetched.get('status')}")

bg = client.responses.create(
    model=DENSE,
    input="Write a short paragraph about idempotency in distributed systems.",
    max_output_tokens=300,
    background=True,
    store=True,
)
print("background job:", redact_ids(bg.id), "status:", bg.status)

for _ in range(30):
    time.sleep(2)
    code, polled = post(
        f"{PREFIX}/responses/{bg.id}", None, region=REGION, method="GET"
    )
    if polled.get("status") in ("completed", "failed", "cancelled"):
        break
print("final status:", polled.get("status"))
print("text:", response_text(polled)[:200])

GET  -> 200 status=completed


background job: resp_4jyx44uu... status: in_progress


final status: completed
text: In distributed systems, **idempotency** is the property of an operation where performing it multiple times has the same effect as performing it once. This is a critical safety mechanism because networ


In [14]:
# Tidy up the stored responses we created.
for rid in (turn1.id, bg.id):
    code, _ = post(f"{PREFIX}/responses/{rid}", None, region=REGION, method="DELETE")
    print(f"DELETE {redact_ids(rid)} -> {code}")

DELETE resp_56ufg573... -> 200


DELETE resp_4jyx44uu... -> 200


## 7. A short detour to Chat Completions

Gemma 4 supports Chat Completions too (same `/openai/v1` prefix). Reach for it
when you have existing OpenAI-shaped code, or want the simpler stateless model.

Two differences worth seeing side by side: `messages` instead of `input`,
`max_completion_tokens` instead of `max_output_tokens` — and **no reasoning
content**.

> **This surface changed on 12 Aug 2026.** Gemma 4 previously accepted
> `max_tokens` here, along with a full `temperature` range and `top_p`. It now
> follows current OpenAI semantics: `max_tokens` is rejected in favour of
> `max_completion_tokens`, `temperature` is pinned to its default, and `top_p`
> is unsupported. The probe in section 3 asks the live endpoint, so you see
> today's answer rather than the one that was true when this was written. Treat
> every parameter table in every sample — including these — as a snapshot.


In [15]:
cc = client.chat.completions.create(
    model=DENSE,
    messages=[
        {"role": "system", "content": "You are terse."},
        {"role": "user", "content": "Why is idempotency useful? One sentence."},
    ],
    max_completion_tokens=120,  # max_tokens is rejected -- see section 3
)
print("content:", cc.choices[0].message.content)
print("\nusage:", cc.usage.model_dump_json())


content: Idempotency prevents unintended side effects or duplicate data when an operation is performed multiple times.

usage: {"completion_tokens":19,"prompt_tokens":47,"total_tokens":66,"completion_tokens_details":{"accepted_prediction_tokens":0,"audio_tokens":0,"reasoning_tokens":0,"rejected_prediction_tokens":0},"prompt_tokens_details":{"audio_tokens":0,"cache_write_tokens":0,"cached_tokens":0}}


In [16]:
# Reasoning still costs tokens on Chat Completions, but the OpenAI CC schema has
# nowhere to return the trace -- you pay for it and cannot read it.
code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": DENSE,
        "messages": [{"role": "user", "content": "Tricky: 17*23?"}],
        "max_completion_tokens": 300,
        "reasoning_effort": "high",
    },
    region=REGION,
)
message = data.get("choices", [{}])[0].get("message", {}) or {}
details = (data.get("usage", {}) or {}).get("completion_tokens_details") or {}
print(f"HTTP {code} | keys in message: {sorted(message.keys())}")
print("reasoning tokens billed:", details.get("reasoning_tokens"))
print("answer:", (message.get("content") or "")[:80])

# Function tools are the one place reasoning must be turned OFF explicitly.
TOOL = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    }
]
print()
for label, extra in [
    ("tools alone", {}),
    ('tools + reasoning_effort "none"', {"reasoning_effort": "none"}),
]:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": DENSE,
            "messages": [{"role": "user", "content": "What is 17*23? Use the tool."}],
            "max_completion_tokens": 200,
            "tools": TOOL,
            **extra,
        },
        region=REGION,
    )
    choice = data.get("choices", [{}])[0].get("message", {}) or {}
    calls = choice.get("tool_calls") or []
    detail = "" if code == 200 else err(data)[:46]
    print(f"  {label:<32} HTTP {code} tool_calls={len(calls)} {detail}")

print()
print('=> Gemma 4 refuses function tools unless reasoning_effort is "none".')


HTTP 200 | keys in message: ['annotations', 'content', 'reasoning', 'refusal', 'role']
reasoning tokens billed: 0
answer: The answer is **391**.

Here is the "trick" to solve it quickly in your head usi



  tools alone                      HTTP 400 tool_calls=0 Function tools with reasoning_effort are not s


  tools + reasoning_effort "none"  HTTP 200 tool_calls=1 

=> Gemma 4 refuses function tools unless reasoning_effort is "none".


## 8. Tool use (function calling)

Gemma 4 has native function calling. Note the **flat** Responses tool shape —
`name`/`description`/`parameters` at the top level, unlike Chat Completions which
nests them under `"function"`.

In [17]:
def get_weather(location: str, unit: str = "celsius") -> dict:
    """Stand-in for a real weather API."""
    table = {"seattle": 12, "singapore": 31, "berlin": 8}
    celsius = table.get(location.split(",")[0].strip().lower(), 20)
    value = celsius if unit == "celsius" else round(celsius * 9 / 5 + 32)
    return {"location": location, "temperature": value, "unit": unit, "sky": "cloudy"}


weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City, e.g. Seattle"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["location"],
    },
}

conv = [{"role": "user", "content": "What's the weather in Seattle?"}]
first = client.responses.create(
    model=DENSE,
    input=conv,
    tools=[weather_tool],
    tool_choice="auto",
    max_output_tokens=300,
)

calls = [i for i in first.output if i.type == "function_call"]
print("tool calls requested:", [(c.name, c.arguments) for c in calls])

for call in calls:
    args = json.loads(call.arguments)
    result = get_weather(**args)
    # Echo the call, then its result. Note: no reasoning items replayed.
    conv.append(
        {
            "type": "function_call",
            "call_id": call.call_id,
            "name": call.name,
            "arguments": call.arguments,
        }
    )
    conv.append(
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result),
        }
    )

final = client.responses.create(
    model=DENSE, input=conv, tools=[weather_tool], max_output_tokens=200
)
print("\nfinal answer:", final.output_text)

tool calls requested: [('get_weather', '{"location":"Seattle"}')]



final answer: The current weather in Seattle is 12°C and cloudy.


### One tool call per turn

Gemma 4's model card states parallel tool calls are not supported. It does not
error — it just quietly does one. Design your loop to iterate rather than
expecting a batch.

In [18]:
tool_a = {
    "type": "function",
    "name": "tool_a",
    "description": "Records a value for A.",
    "parameters": {
        "type": "object",
        "properties": {"x": {"type": "string"}},
        "required": ["x"],
    },
}
tool_b = {
    "type": "function",
    "name": "tool_b",
    "description": "Records a value for B.",
    "parameters": {
        "type": "object",
        "properties": {"y": {"type": "string"}},
        "required": ["y"],
    },
}

multi = client.responses.create(
    model=DENSE,
    input="Call tool_a with x='1' AND tool_b with y='2'. Both, right now.",
    tools=[tool_a, tool_b],
    max_output_tokens=300,
)
issued = [i.name for i in multi.output if i.type == "function_call"]
print(f"asked for 2 tool calls, model issued {len(issued)}: {issued}")
print("=> iterate; do not assume a batch.")

asked for 2 tool calls, model issued 1: ['tool_a']
=> iterate; do not assume a batch.


## 9. Structured JSON output

Two routes, and an important caveat about the first one.

In [19]:
# (a) Native strict schema via text.format
schema = {
    "type": "object",
    "properties": {
        "language": {"type": "string"},
        "typed": {"type": "boolean"},
        "year_created": {"type": "integer"},
    },
    "required": ["language", "typed", "year_created"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Describe the Rust programming language.",
        "max_output_tokens": 200,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "lang",
                "schema": schema,
                "strict": True,
            }
        },
    },
    region=REGION,
)
raw = response_text(data)
print("native json_schema ->", code)
print("raw output:", repr(raw))

native json_schema -> 200
raw output: '{"language":"Rust","typed":true,"year_created":2010}'


### ⚠️ "Strict" is not reliably strict on Gemma 4

Gemma 4 intermittently appends characters **after** a well-formed JSON object —
a stray `\n}`, or occasionally unrelated text. The object itself is correct, but
`json.loads()` on the whole string raises. In repeated testing this happened in
roughly half of runs.

Never call bare `json.loads()` on Gemma 4 structured output in production.

In [20]:
runs, invalid = 6, 0
for i in range(runs):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "Describe the Rust programming language.",
            "max_output_tokens": 200,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "lang",
                    "schema": schema,
                    "strict": True,
                }
            },
        },
        region=REGION,
    )
    text = response_text(data)
    try:
        json.loads(text)
        verdict = "parses"
    except json.JSONDecodeError:
        verdict = "FAILS json.loads"
        invalid += 1
    print(f"  run {i + 1}: {verdict:18} {text!r}")

print(f"\n{invalid}/{runs} runs would crash a naive json.loads()")

  run 1: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 2: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 3: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 4: parses             '{"language":"Rust","typed":true,"year_created":2010}\n'


  run 5: FAILS json.loads   '{"language":"Rust","typed":true,"year_created":2010}\n}'


  run 6: parses             '{"language":"Rust","typed":true,"year_created":2010}\n'

1/6 runs would crash a naive json.loads()


In [21]:
# The fix: extract the first balanced JSON object. bedrock.parse_json_lenient()
# does exactly this, and is safe to use on every model.

for sample in [
    '{"language":"Rust","typed":true,"year_created":2010}',
    '{"language":"Rust","typed":true,"year_created":2010}\n}',
    '{"language":"Rust","typed":true,"year_created":2010}\ntrailing text',
    '```json\n{"language":"Rust","typed":true,"year_created":2010}\n```',
]:
    print(f"  {parse_json_lenient(sample)}   <- from {sample[:52]!r}")

  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '```json\n{"language":"Rust","typed":true,"year_create'


In [22]:
# Re-run the real call and parse it safely.
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Describe the Rust programming language.",
        "max_output_tokens": 200,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "lang",
                "schema": schema,
                "strict": True,
            }
        },
    },
    region=REGION,
)
parsed = parse_json_lenient(response_text(data))
print("parsed safely:")
print(json.dumps(parsed, indent=2))
expected = {"language", "typed", "year_created"}
if set(parsed) != expected:
    raise ValueError(
        f"schema mismatch: expected {sorted(expected)}, got {sorted(parsed)}"
    )
print("schema honoured exactly:", sorted(parsed))

parsed safely:
{
  "language": "Rust",
  "typed": true,
  "year_created": 2010
}
schema honoured exactly: ['language', 'typed', 'year_created']


In [23]:
# (b) Forced tool call — the arguments ARE the output. More portable: it works on
# models that lack native structured output, and enum constraints are respected.
emit = {
    "type": "function",
    "name": "emit_profile",
    "description": "Return the analysis. Use 'unknown' if unsure.",
    "parameters": {
        "type": "object",
        "properties": {
            "summary": {"type": "string"},
            "sentiment": {
                "type": "string",
                "enum": ["positive", "neutral", "negative"],
            },
            "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        },
        "required": ["summary", "sentiment", "confidence"],
    },
}

forced = client.responses.create(
    model=DENSE,
    input="Review: 'The battery life is superb but the screen scratches easily.'",
    tools=[emit],
    tool_choice={"type": "function", "name": "emit_profile"},  # must call it
    max_output_tokens=300,
)
call = next(i for i in forced.output if i.type == "function_call")
print("forced-tool output:")
# parse_json_lenient again: tool arguments can carry the same trailing garbage.
print(json.dumps(parse_json_lenient(call.arguments), indent=2))

forced-tool output:
{
  "summary": "The user is pleased with the battery life but dissatisfied with the screen's durability. sun",
  "sentiment": "neutral",
  "confidence": "high"
}


### Schema keywords

You may read that Gemma 4 rejects JSON-Schema constraint keywords like
`minLength` and `pattern`. On the current `/openai/v1` Responses path they are
**accepted** — verify against your own path and model before adding a sanitiser.

In [24]:
constrained = {
    "type": "function",
    "name": "emit",
    "description": "Emit a code.",
    "parameters": {
        "type": "object",
        "properties": {"code": {"type": "string", "minLength": 3, "pattern": "^[A-Z]"}},
        "required": ["code"],
    },
}
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Emit code 'ABC'.",
        "max_output_tokens": 100,
        "tools": [constrained],
    },
    region=REGION,
)
print(
    f"schema with minLength + pattern -> HTTP {code} "
    f"{'accepted' if code == 200 else err(data)[:70]}"
)

schema with minLength + pattern -> HTTP 200 accepted


## 10. Multimodal — image input

All three variants take text + image. Constraints from the model card and
testing:

- Base64 data URLs or `s3://` URLs. **Arbitrary `https://` image URLs are not
  supported.**
- Total request body max **3.5 MB**.
- Put the image **before** the text (Google's recommended ordering).
- Very small images are rejected as an unsupported format — use realistic sizes.

In [25]:
import struct
import zlib


def make_png(width: int, height: int, rgb: tuple) -> bytes:
    """Build a solid-colour PNG without external dependencies."""
    raw = b"".join(b"\x00" + bytes(rgb) * width for _ in range(height))

    def chunk(tag: bytes, payload: bytes) -> bytes:
        body = tag + payload
        return (
            struct.pack(">I", len(payload))
            + body
            + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)
        )

    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
        + chunk(b"IDAT", zlib.compress(raw))
        + chunk(b"IEND", b"")
    )


crimson_png = make_png(64, 64, (220, 20, 60))
data_url = "data:image/png;base64," + base64.b64encode(crimson_png).decode()
print("data URL bytes:", len(data_url))

vision = client.responses.create(
    model=DENSE,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_image", "image_url": data_url},  # image FIRST
                {"type": "input_text", "text": "What colour is this image? One word."},
            ],
        }
    ],
    max_output_tokens=32,
)
print("model sees:", repr(vision.output_text.strip()))

data URL bytes: 262


model sees: 'Red'


In [26]:
# A 1x1 pixel PNG is rejected — worth knowing so you don't chase a phantom bug.
tiny = "data:image/png;base64," + base64.b64encode(make_png(1, 1, (255, 0, 0))).decode()
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "max_output_tokens": 16,
        "input": [
            {
                "role": "user",
                "content": [
                    {"type": "input_image", "image_url": tiny},
                    {"type": "input_text", "text": "Colour?"},
                ],
            }
        ],
    },
    region=REGION,
)
print(f"1x1 PNG -> HTTP {code}: {err(data)[:80]}")

1x1 PNG -> HTTP 200: {"background": false, "billing": {"payer": "developer"}, "completed_at": 1786701


There is also an undocumented ceiling of roughly **32 images per request**.
Payloads far below the 3.5 MB size limit can still fail with
`400 / "Engine bad request"` once you exceed it. Batch large image sets.

## 11. Choosing a variant — a like-for-like comparison

Same prompt, all three variants, measuring latency and tokens.

In [27]:
task = "In one sentence, explain why eventual consistency is a useful trade-off."

print(f"{'model':26} {'latency':>9} {'reason tok':>11} {'out tok':>8}  answer")
print("-" * 104)
for model in (COMPACT, MOE, DENSE):
    started = time.perf_counter()
    r = client.responses.create(
        model=model, input=task, reasoning={"effort": "low"}, max_output_tokens=250
    )
    elapsed = time.perf_counter() - started
    details = r.usage.output_tokens_details
    print(
        f"{model:26} {elapsed:>8.2f}s {details.reasoning_tokens:>11} "
        f"{r.usage.output_tokens:>8}  {r.output_text.strip()[:44]!r}"
    )

model                        latency  reason tok  out tok  answer
--------------------------------------------------------------------------------------------------------


google.gemma-4-e2b             0.69s           0       37  'Eventual consistency is a useful trade-off b'


google.gemma-4-26b-a4b         0.61s           0       35  'Eventual consistency allows for higher avail'


google.gemma-4-31b             0.91s           0       34  'Eventual consistency is a useful trade-off b'


| If your workload is… | Choose | Why |
|---|---|---|
| Reasoning- or coding-heavy | `gemma-4-31b` | Largest dense variant, 256K context |
| Cost-sensitive at high throughput | `gemma-4-26b-a4b` | MoE (mixture-of-experts): ~4B-class cost, larger knowledge capacity |
| Latency-sensitive, on-device-style | `gemma-4-e2b` | Smallest and fastest; set `effort="high"` |

All three share one API surface, so you can develop once and switch by model ID.

## 12. Production hardening

Retries, cost attribution, and privacy in one place.
(Background: `../00-foundations/03-scaling-tiers-and-latency.ipynb`.)

In [28]:
# Attribute usage to a project for cost tracking (see ../00-foundations/02).
code, project = post(
    "/v1/organization/projects",
    {
        "name": "gemma4-samples",
        "tags": {"Application": "Gemma4Demo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Reply OK",
        "max_output_tokens": 16,
        "service_tier": "flex",
        "store": False,
    },
    region=REGION,
    headers={"OpenAI-Project": project_id},
)
print(f"attributed call -> HTTP {code} resolved tier={data.get('service_tier')}")

project: 200 proj_6vveqcmw...


attributed call -> HTTP 200 resolved tier=flex


In [29]:
class Gemma4Client:
    """Teaching pattern: fresh token, right params, retries, attribution.
    Not production-ready as written - review and adapt before deployment."""

    def __init__(self, model=DENSE, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def ask(self, prompt, *, effort="low", max_output_tokens=512, structured=None):
        body = {
            "model": self.model,
            "input": prompt,
            "max_output_tokens": max(16, max_output_tokens),  # API minimum is 16
            "temperature": 1.0,  # the ONLY value Responses accepts
            # top_p deliberately omitted: rejected on the Responses API
            "reasoning": {"effort": effort},
            "service_tier": self.tier,
            "store": False,  # no 30-day retention
        }
        if structured:
            body["text"] = {
                "format": {
                    "type": "json_schema",
                    "name": "out",
                    "schema": structured,
                    "strict": True,
                }
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429/5xx with exponential backoff — mantle has no RPM
        # quota and sheds load under regional pressure.
        code, data = post(
            f"{PREFIX}/responses", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data


gemma = Gemma4Client(model=MOE, tier="flex", project=project_id)
out = gemma.ask(
    "Name the capital of Japan.",
    structured={
        "type": "object",
        "properties": {"capital": {"type": "string"}},
        "required": ["capital"],
        "additionalProperties": False,
    },
)
print("structured:", parse_json_lenient(response_text(out)))
print("usage:", json.dumps(out.get("usage", {})))

structured: {'capital': 'Tokyo'}
usage: {"input_tokens": 72, "input_tokens_details": {"cache_write_tokens": 0, "cached_tokens": 0}, "output_tokens": 6, "output_tokens_details": {"reasoning_tokens": 0}, "total_tokens": 78}


In [30]:
# Clean up the demo project.
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived project:", code, archived.get("status"))

archived project: 200 archived


## Gotchas — Gemma 4 on bedrock-mantle

| Gotcha | Detail |
|---|---|
| `bedrock-mantle` only | No `bedrock-runtime` support for these model IDs |
| Path prefix | `/openai/v1`, **not** the bare `/v1` most mantle models use |
| `temperature` | On Responses, **only `1.0` is accepted** — every other value 400s |
| `top_p` | **400 on Responses**; accepted on Chat Completions |
| Tuning sampling | Not possible on Responses — switch to Chat Completions |
| `max_output_tokens` | Minimum **16**; smaller values 400 |
| `reasoning.effort` | `none`/`low`/`medium`/`high`; **`minimal` is rejected** |
| Reasoning visibility | Returned on Responses only; billed-but-hidden on Chat Completions |
| Reasoning replay | Never append reasoning items to history — degrades quality |
| Parallel tool calls | Unsupported; model silently issues one |
| `store=False` | Blocks `previous_response_id` chaining (404) |
| Images | base64 or `s3://` only; ≤3.5 MB body; ~32 images max; 1×1 PNG rejected |
| e2b reasoning | Set `effort="high"` to stop thinking leaking into the answer |
| Regions | The only family in all four mantle Regions |

## Where next
- Same-API neighbours: `../01-openai-gpt/` (web search, caching), `../11-xai-grok/`
- Different API shape: `../04-qwen/` (Chat Completions), `../02-anthropic-claude/`
  (Messages)
- Cross-cutting: `../99-cross-cutting/`

## Also on `bedrock-runtime`? Gemma 4 — no

Gemma 4 is **`bedrock-mantle` only**. There is no Converse path for it today, so a workload on Gemma 4 cannot be moved to `bedrock-runtime` without changing model. Gemma **3** is on both endpoints - see the sibling notebook.

The cell below confirms it against the live catalogues rather than asserting it,
because model availability moves.


In [31]:
from bedrock import endpoints_for, runtime_models

MODEL = "google.gemma-4-31b"
where = endpoints_for(MODEL)
print(f"{MODEL} -> {where}")

if not where["runtime"]:
    print("\nNot on bedrock-runtime, so Converse is not an option for this model.")
    print("Same-provider models that ARE on bedrock-runtime today:")
    provider = MODEL.split(".")[0]
    siblings = sorted(m for m in runtime_models() if m.startswith(provider + "."))
    for sibling in siblings[:8]:
        print("   ", sibling)
    if not siblings:
        print("    (none)")


google.gemma-4-31b -> {'mantle': True, 'runtime': False}

Not on bedrock-runtime, so Converse is not an option for this model.
Same-provider models that ARE on bedrock-runtime today:


    google.gemma-3-12b-it
    google.gemma-3-27b-it
    google.gemma-3-4b-it
